# 20.19 — Cost & latency optimization

Cost and latency optimization finds the cheapest path that still satisfies quality and tail-latency promises.

System design supplies budgets, while serving, quantization, and ANN supply the knobs. Edge ML is the strongest version of the same budget discipline. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 17
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)

## The concept, built once

Expected latency and request cost: $E[L]=hL_h+(1-h)L_m$ and $\text{cost}=nc$.

The reusable policy evaluates cache, model tier, and batching choices, then returns the cheapest candidate that satisfies a p95 latency constraint.

In [ ]:
def pctl_nearest(values, q):
    sorted_values = np.sort(np.asarray(values, dtype=float))
    index = int(np.ceil(q * len(sorted_values))) - 1
    index = min(max(index, 0), len(sorted_values) - 1)
    return float(sorted_values[index])

def latency_cost_policy(requests, hit_rate=0.60, unit_cost=0.00003, p95_target=180, batch_size=1, max_wait_ms=0):
    base_latency = np.asarray(requests["base_latency"], dtype=float)
    cacheable = np.asarray(requests["cacheable"], dtype=bool)
    difficulty = np.asarray(requests["difficulty"], dtype=float)
    hits = cacheable & (difficulty <= hit_rate)
    model_latency = np.where(difficulty < 0.70, 10, 60)
    cache_latency = np.where(hits, 4, 80)
    compute_latency = np.maximum(model_latency, cache_latency)
    queue_wait = np.full_like(compute_latency, max_wait_ms * (batch_size - 1) / max(batch_size, 1), dtype=float)
    latency = base_latency + compute_latency + queue_wait
    cost = np.where(difficulty < 0.70, unit_cost, unit_cost * 5)
    total_cost = float(np.sum(cost))
    p95 = pctl_nearest(latency, 0.95)
    feasible = p95 <= p95_target
    return {"latency": latency, "p95": p95, "cost": total_cost, "feasible": feasible, "hits": hits}

lesson_latencies = np.array([40, 55, 60, 68, 72, 72, 90, 110, 150, 180], dtype=float)
lesson_p50 = pctl_nearest(lesson_latencies, 0.50)
lesson_p95 = pctl_nearest(lesson_latencies, 0.95)
ratio = lesson_p95 / lesson_p50
batch_compute = 40 / 8
cache_expected = 0.60 * 4 + 0.40 * 80
tier_expected = 0.70 * 10 + 0.30 * 60
cloud_cost = 1000000 * 0.00003
assert lesson_p95 == 180
assert lesson_p50 == 72
assert abs(ratio - 2.5) < 1e-12
assert batch_compute == 5
assert abs(cache_expected - 34.4) < 1e-12
assert abs(tier_expected - 25) < 1e-12
assert abs(cloud_cost - 30) < 1e-12
print("p95/p50", ratio)
print("amortized compute", batch_compute)
print("cache expected latency", cache_expected)

The same method now becomes the single evaluation API. It returns the operational artifact and the topic metric so the D1 proof scales to D2–D5.

In [ ]:
print("Build-once assertions passed for 20.19")

## The dataset ladder

Family F17 uses an inline D1–D5 systems workload ladder. Each rung increases operational realism while keeping the computation CPU-only, seeded, and NumPy based.

In [ ]:
def make_requests(n, rng, base_mean, tail_rate=0.0, diurnal=False):
    base = rng.gamma(3, base_mean / 3, size=n)
    if diurnal:
        wave = 1 + 0.45 * np.sin(np.linspace(0, 4 * np.pi, n))
        base = base * wave
    tail = rng.random(n) < tail_rate
    base[tail] = base[tail] + rng.gamma(2, 80, size=np.sum(tail))
    cacheable = rng.random(n) < 0.65
    difficulty = rng.random(n)
    arrival = np.cumsum(rng.exponential(1 / 1000, size=n))
    return {"base_latency": base, "cacheable": cacheable, "difficulty": difficulty, "arrival": arrival}

def build_latency_ladder(seed=17):
    rng = np.random.default_rng(seed)
    ladder = []
    ladder.append({"name": "D1 ten request latencies", "requests": {"base_latency": lesson_latencies, "cacheable": np.array([True] * 10), "difficulty": np.linspace(0.1, 0.9, 10), "arrival": np.arange(10)}, "target": 220, "load": 10})
    ladder.append({"name": "D2 stable cacheable 100k", "requests": make_requests(100000, rng, 20), "target": 180, "load": 100000})
    ladder.append({"name": "D3 tail spikes and queueing", "requests": make_requests(120000, rng, 24, 0.06), "target": 190, "load": 120000})
    ladder.append({"name": "D4 diurnal load trace", "requests": make_requests(140000, rng, 22, 0.03, True), "target": 185, "load": 140000})
    ladder.append({"name": "D5 multi-tier routing simulation", "requests": make_requests(180000, rng, 28, 0.08, True), "target": 200, "load": 180000})
    return ladder

ladder = build_latency_ladder()
for rung in ladder:
    req = rung["requests"]
    print(rung["name"], "n", len(req["base_latency"]), "target", rung["target"], "base sample", np.round(req["base_latency"][:5], 1).tolist())

## Run the same method across D1–D5

The metric is collected in the same way for every rung, so changes across the ladder reflect workload complexity rather than a different measurement.

In [ ]:
results = []
for rung in ladder:
    candidates = []
    for hit_rate in [0.45, 0.60, 0.75]:
        for batch_size in [1, 4, 8]:
            max_wait = 0 if batch_size == 1 else 6
            result = latency_cost_policy(rung["requests"], hit_rate=hit_rate, p95_target=rung["target"], batch_size=batch_size, max_wait_ms=max_wait)
            result["hit_rate"] = hit_rate
            result["batch_size"] = batch_size
            candidates.append(result)
    feasible = [candidate for candidate in candidates if candidate["feasible"]]
    if feasible:
        chosen = min(feasible, key=lambda x: x["cost"])
    else:
        chosen = min(candidates, key=lambda x: x["p95"])
    results.append({"name": rung["name"], "metric": chosen["cost"], "artifact": chosen, "load": rung["load"], "target": rung["target"]})

print("rung | cost at p95 constraint | p95 | batch")
for item in results:
    artifact = item["artifact"]
    print(item["name"], round(item["metric"], 3), round(artifact["p95"], 2), artifact["batch_size"])

## Results visualization

The closing figure has operational panels for each rung plus a metric-vs-load curve.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for col, item in enumerate(results):
    artifact = item["artifact"]
    axes[0, col].hist(artifact["latency"][:5000], bins=40, color="tab:cyan")
    axes[0, col].axvline(item["target"], color="black", linestyle="--")
    axes[0, col].set_title(item["name"].split()[0] + " latency")
    axes[0, col].set_xlabel("ms")
    hit_series = np.cumsum(artifact["hits"][:1000]) / np.arange(1, 1001)
    axes[1, col].plot(hit_series, color="tab:olive")
    axes[1, col].set_xlabel("request")
    axes[1, col].set_ylabel("cache hit rate")
fig.suptitle("Operational panels: latency tails and cache behavior")
plt.tight_layout()

fig, ax = plt.subplots(figsize=(7, 4))
loads = [item["load"] for item in results]
metrics = [item["metric"] for item in results]
ax.plot(loads, metrics, marker="o")
ax.set_xlabel("load")
ax.set_ylabel("cost under p95 constraint")
ax.set_title("Metric vs load")
plt.show()

## Pitfall on the hardest rung

Reproduce the lesson pitfall on D5, then apply the operational fix and compare the metric.

In [ ]:
d5 = ladder[-1]
no_queue = latency_cost_policy(d5["requests"], batch_size=8, max_wait_ms=0, p95_target=d5["target"])
with_queue = latency_cost_policy(d5["requests"], batch_size=8, max_wait_ms=20, p95_target=d5["target"])
fixed = latency_cost_policy(d5["requests"], batch_size=4, max_wait_ms=4, p95_target=d5["target"])
print("wrong batched p95 without queue", round(no_queue["p95"], 2))
print("actual batched p95 with queue", round(with_queue["p95"], 2))
print("fixed max-wait p95", round(fixed["p95"], 2))

## Evaluate it + Practice

- Metric: cost at a p95-latency constraint, compared with a cheapest-route baseline.
- Sanity check: p95 180 ms divided by p50 72 ms is 2.5x.
- Ablation: remove queue wait from batching and compare predicted vs actual p95.
- Failure signals: p95 target violations, cost cliffs, and cache hit-rate drift.

Practice prompts:
1. Change one workload knob on D3 and predict whether the metric should improve or degrade.
2. Add one slice or route to D4 and explain which operational panel should change.
3. Tighten the D5 budget or threshold and rerun only after saving your own copy.